# A/B Test Analysis Workflow

This notebook performs an end-to-end A/B test analysis including:
1. Data validation & Quality checks (SRM test)
2. Conversion Rate (CR) analysis (Z-test / Chi-Square)
3. Revenue / ARPU analysis (t-test, Mann-Whitney U, Bootstrap)
4. Decision making & Visualizations

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.stats.api as sms
import seaborn as sns
import matplotlib.pyplot as plt

# Set plots style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Data & Initial Inspection

In [ ]:
# Replace with your dataset loading logic (e.g., pd.read_csv('ab_test_data.csv'))
# Expected columns: user_id, group ('A' or 'B'), converted (0 or 1), revenue (float)

# Generating synthetic data for demonstration
np.random.seed(42)
n_a, n_b = 10000, 10000

df_a = pd.DataFrame({
    'user_id': range(1, n_a + 1),
    'group': 'A',
    'converted': np.random.binomial(1, 0.10, n_a),
    'revenue': np.random.exponential(scale=15, size=n_a)
})
df_a['revenue'] = df_a['revenue'] * df_a['converted']

df_b = pd.DataFrame({
    'user_id': range(n_a + 1, n_a + n_b + 1),
    'group': 'B',
    'converted': np.random.binomial(1, 0.115, n_b), # 1.5% uplift
    'revenue': np.random.exponential(scale=16, size=n_b)
})
df_b['revenue'] = df_b['revenue'] * df_b['converted']

df = pd.concat([df_a, df_b], ignore_index=True)
df.head()

## 2. Sample Ratio Mismatch (SRM) Check
Check if the traffic split between variants matches the expected ratio (e.g., 50/50).

In [ ]:
observed = df['group'].value_counts()
total_users = observed.sum()
expected = [total_users / 2, total_users / 2]  # Assuming 50/50 split

chi2_stat, p_val_srm = stats.chisquare(f_obs=observed, f_exp=expected)
print(f"SRM Check P-Value: {p_val_srm:.4f}")

if p_val_srm < 0.01:
    print("WARNING: Sample Ratio Mismatch detected! Investigate traffic assignment.")
else:
    print("SUCCESS: No SRM detected. Traffic split is balanced.")

## 3. Conversion Rate (CR) Analysis
Compare conversion rates using a two-proportion Z-test and Chi-Square test.

In [ ]:
summary_cr = df.groupby('group')['converted'].agg(['count', 'sum', 'mean']).rename(columns={'mean': 'conversion_rate'})
summary_cr['conversion_rate_pct'] = summary_cr['conversion_rate'] * 100
print(summary_cr)

# Proportions Z-Test
count_conv = summary_cr['sum']
n_obs = summary_cr['count']
z_stat, cr_p_val = sms.proportions_ztest(count_conv, n_obs)

cr_a = summary_cr.loc['A', 'conversion_rate']
cr_b = summary_cr.loc['B', 'conversion_rate']
relative_lift = ((cr_b - cr_a) / cr_a) * 100

print(f"\n--- Conversion Analysis Result ---")
print(f"Relative Lift: {relative_lift:.2f}%")
print(f"Z-Test P-Value: {cr_p_val:.4f}")

if cr_p_val < 0.05:
    print("Statistically Significant Result: Reject the Null Hypothesis.")
else:
    print("Not Statistically Significant: Fail to Reject the Null Hypothesis.")

## 4. Revenue & ARPU Analysis
Evaluate Average Revenue Per User (ARPU) using T-Test and Bootstrap.

In [ ]:
rev_a = df[df['group'] == 'A']['revenue']
rev_b = df[df['group'] == 'B']['revenue']

arpu_a = rev_a.mean()
arpu_b = rev_b.mean()

print(f"ARPU Group A: ${arpu_a:.2f}")
print(f"ARPU Group B: ${arpu_b:.2f}")

# Two-sample Welch's t-test (handles unequal variances)
t_stat, arpu_p_val = stats.ttest_ind(rev_a, rev_b, equal_var=False)
print(f"Welch's T-Test P-Value: {arpu_p_val:.4f}")

# Mann-Whitney U Test (Non-parametric alternative for skewed revenue distribution)
mw_stat, mw_p_val = stats.mannwhitneyu(rev_a, rev_b)
print(f"Mann-Whitney U P-Value: {mw_p_val:.4f}")

## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Conversion Rate Barplot
sns.barplot(data=df, x='group', y='converted', ci=95, ax=axes[0], palette='Blues')
axes[0].set_title('Conversion Rate by Group (with 95% CI)')
axes[0].set_ylabel('Conversion Rate')

# Revenue Distribution (paying users only)
paying_df = df[df['revenue'] > 0]
sns.boxplot(data=paying_df, x='group', y='revenue', ax=axes[1], palette='Greens')
axes[1].set_title('Revenue Distribution (Paying Users Only)')
axes[1].set_ylabel('Revenue ($)')

plt.tight_layout()
plt.show()